In [67]:
import random
def creat_grid(size : int):
  #creation dune gride vide
  grid=[['-'for i in range (size)]for j in range(size)]
  # inisialisation de pos de depart
  grid[0][0]="S"
  grid[size-1][size-1]="T"
  #definition de trois obstacles
  nb_obs=2
  count=0
  while (count<nb_obs):
    x=random.randint(0,size-1)
    y=random.randint(0,size-1)
    if(grid[x][y]=="-"):
      grid[x][y]="O"
      count+=1
  return grid

In [68]:
import random
def create_bandwidth(grid):
    size = len(grid)
    bandwidth = []
    for i in range(size):
        row = []
        for j in range(size):
            # obstacle
            if grid[i][j] == 'O':
                row.append(0)
            else:
                row.append(random.randint(1,10))
        bandwidth.append(row)
    return bandwidth

In [69]:
import random
def create_distance(grid):
    size = len(grid)
    distance = []
    for i in range(size):
        row = []
        for j in range(size):
            # obstacle
            if grid[i][j] == 'O':
                row.append(0)
            else:
                row.append(random.randint(1,10))
        distance.append(row)
    return distance

In [70]:
liste=creat_grid(3)
bandwidth = create_bandwidth(liste)
distance = create_distance(liste)

def display(grid):
  print("Grid:")
  for i in grid:
    print("  ".join(i))
  print("—————————————————————")

display(liste)

def display_bandwidth(bandwidth):
  print("Bandwidth:")
  for row in bandwidth:
        print(row)
  print("—————————————————————")

display_bandwidth(bandwidth)

def display_distance(distance):
  print("Distance:")
  for row in distance:
        print(row)
  print("—————————————————————")

display_distance(distance)

def display_cost_matrix(distance, bandwidth):
    size = len(distance)
    print("Cost Matrix:")
    for i in range(size):
        row = []
        for j in range(size):
            # obstacle
            if bandwidth[i][j] == 0:
                row.append("O")
            else:
                cost = distance[i][j] / bandwidth[i][j]
                row.append(round(cost,2))
        print(row)
display_cost_matrix(distance, bandwidth)


Grid:
S  O  -
-  -  O
-  -  T
—————————————————————
Bandwidth:
[7, 0, 8]
[1, 7, 0]
[6, 9, 4]
—————————————————————
Distance:
[6, 0, 6]
[8, 9, 0]
[8, 8, 1]
—————————————————————
Cost Matrix:
[0.86, 'O', 0.75]
[8.0, 1.29, 'O']
[1.33, 0.89, 0.25]


In [71]:
def move_agent(grid, start, target):

    x, y = start
    tx, ty = target
    size = len(grid)
    # directions possibles
    directions = []
    if x < tx:
        directions.append((x+1, y))
    if y < ty:
        directions.append((x, y+1))
    if x > tx:
        directions.append((x-1, y))
    if y > ty:
        directions.append((x, y-1))
    # essayer chaque direction
    for new_x, new_y in directions:
        if (0 <= new_x < size and
            0 <= new_y < size and
            grid[new_x][new_y] != 'O'):
            return (new_x, new_y)

    return start

In [72]:
def heuristic(current, target):

    x1, y1 = current
    x2, y2 = target

    return abs(x1 - x2) + abs(y1 - y2)

In [73]:
import heapq

def astar(grid,bandwidth,distance):
    size = len(grid)
    start = (0,0)
    target = (size-1,size-1)
    # liste open
    open_list = []
    # ajouter le noeud initial
    heapq.heappush(open_list, (0, start))
    # dictionnaire des parents
    parent = {}
    # cout reel
    g_cost = {}
    g_cost[start] = 0
    # noeuds visites
    close_list = set()
    while open_list:
        # recuperer le noeud avec plus petit f(n)
        f, current = heapq.heappop(open_list)
        # si cible atteinte
        if current == target:
            path = []
            while current in parent:
                path.append(current)
                current = parent[current]
            path.append(start)
            path.reverse()
            return path
        close_list.add(current)
        x, y = current
        # directions possibles
        directions = [
            (1,0),
            (-1,0),
            (0,1),
            (0,-1)
        ]
        for dx, dy in directions:
            new_x = x + dx
            new_y = y + dy
            neighbor = (new_x, new_y)
            # verification des limites
            if (0 <= new_x < size and
                0 <= new_y < size):
                # obstacle
                if grid[new_x][new_y] == 'O':
                    continue
                # deja visite
                if neighbor in close_list:
                    continue
                # cout du deplacement
                move_cost = distance[new_x][new_y] / bandwidth[new_x][new_y]
                # calcul g(n)
                new_g = g_cost[current] + move_cost
                # si nouveau chemin meilleur
                if neighbor not in g_cost or new_g < g_cost[neighbor]:
                    g_cost[neighbor] = new_g
                    # heuristique
                    h = heuristic(neighbor, target)
                    # calcul f(n)
                    f = new_g + h
                    # ajouter dans open
                    heapq.heappush(open_list, (f, neighbor))
                    # sauvegarder parent
                    parent[neighbor] = current

    return None

In [74]:
import time

def simulate_path(grid, path):

    if path is None:
        print("No path found")
        return

    for step in path:

        x, y = step

        # ne pas remplacer S ou T
        if grid[x][y] not in ['S', 'T']:
            grid[x][y] = '*'

        display(grid)

        time.sleep(1)

In [75]:
path = astar(liste,bandwidth,distance)
print(path)
simulate_path(liste, path)

[(0, 0), (1, 0), (1, 1), (2, 1), (2, 2)]
Grid:
S  O  -
-  -  O
-  -  T
—————————————————————
Grid:
S  O  -
*  -  O
-  -  T
—————————————————————
Grid:
S  O  -
*  *  O
-  -  T
—————————————————————
Grid:
S  O  -
*  *  O
-  *  T
—————————————————————
Grid:
S  O  -
*  *  O
-  *  T
—————————————————————
